
#### refunder agent

this notebook creates an agent with tools to suggest refunds for orders

#### Tool & View Registration

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ${CATALOG}.ai;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION ${CATALOG}.ai.get_order_details(oid STRING COMMENT 'order id of the order')
RETURNS TABLE (
  body STRING COMMENT 'Body of the event',
  event_type STRING COMMENT 'The type of event',
  order_id STRING COMMENT 'The order id',
  ts STRING COMMENT 'The timestamp of the event',
  location STRING COMMENT 'the location of the order'
)
COMMENT 'Returns all events associated with the order id (oid)'
RETURN
  SELECT ae.body, ae.event_type, ae.order_id, ae.ts, loc.name as location
  FROM ${CATALOG}.lakeflow.all_events ae
  LEFT JOIN ${CATALOG}.simulator.locations loc ON ae.location_id = loc.location_id
  WHERE ae.order_id = oid;

In [0]:
%sql
CREATE OR REPLACE FUNCTION ${CATALOG}.ai.get_order_delivery_time(oid STRING COMMENT 'order id of the order')
RETURNS TABLE (
  order_id STRING COMMENT 'The order id',
  creation_time TIMESTAMP COMMENT 'The timestamp of the first event for the order',
  delivery_time TIMESTAMP COMMENT 'The timestamp of the last event for the order',
  duration_minutes FLOAT COMMENT 'The total duration from the first to the last event in minutes'
)
COMMENT 'Returns the first event time, last event time, and total duration for a given order id.'
RETURN
  WITH MinMaxTimestamps AS (
    SELECT
      MIN(try_to_timestamp(ts)) as first_event_time,
      MAX(try_to_timestamp(ts)) as last_event_time
    FROM
      ${CATALOG}.lakeflow.all_events
    WHERE
      order_id = oid
  )
  SELECT
    oid as order_id,
    first_event_time AS creation_time,
    last_event_time AS delivery_time,
    CAST(
      try_divide(
        (UNIX_TIMESTAMP(last_event_time) - UNIX_TIMESTAMP(first_event_time)),
        60
      ) AS FLOAT
    ) AS duration_minutes
  FROM
    MinMaxTimestamps;

In [ ]:
%sql
CREATE OR REPLACE VIEW ${CATALOG}.ai.order_delivery_times_per_location_view AS
WITH order_times AS (
  SELECT
    ae.order_id,
    loc.name as location,
    MAX(CASE WHEN ae.event_type = 'order_created' THEN try_to_timestamp(ae.ts) END) AS order_created_time,
    MAX(CASE WHEN ae.event_type = 'delivered' THEN try_to_timestamp(ae.ts) END) AS delivered_time
  FROM
    ${CATALOG}.lakeflow.all_events ae
  LEFT JOIN ${CATALOG}.simulator.locations loc ON ae.location_id = loc.location_id
  WHERE
    try_to_timestamp(ae.ts) >= CURRENT_TIMESTAMP() - INTERVAL 1 DAY
  GROUP BY
    ae.order_id,
    loc.name
),
total_order_times AS (
  SELECT
    order_id,
    location,
    (UNIX_TIMESTAMP(delivered_time) - UNIX_TIMESTAMP(order_created_time)) / 60 AS total_order_time_minutes
  FROM
    order_times
  WHERE
    order_created_time IS NOT NULL
    AND delivered_time IS NOT NULL
)
SELECT
  location,
  PERCENTILE(total_order_time_minutes, 0.50) AS P50,
  PERCENTILE(total_order_time_minutes, 0.75) AS P75,
  PERCENTILE(total_order_time_minutes, 0.99) AS P99
FROM
  total_order_times
GROUP BY
  location

In [0]:
%sql
CREATE OR REPLACE FUNCTION ${CATALOG}.ai.get_location_timings(loc STRING COMMENT 'Location name as a string')
RETURNS TABLE (
  location STRING COMMENT 'Location of the order source',
  P50 FLOAT COMMENT '50th percentile',
  P75 FLOAT COMMENT '75th percentile',
  P99 FLOAT COMMENT '99th percentile'
)
COMMENT 'Returns the 50/75/99th percentile of total delivery times for locations'
RETURN
  SELECT location, P50, P75, P99
  FROM ${CATALOG}.ai.order_delivery_times_per_location_view AS odlt
  WHERE odlt.location = loc;

In [ ]:
%sql
-- USE CATALOG is needed in addition to USE SCHEMA + EXECUTE so the serving
-- endpoint's auto-generated SP can traverse the catalog to reach the UC
-- functions at model-load time.  Normally granted by the root data stage
-- (canonical_data/raw_data), but repeated here so this stage is self-
-- sufficient if run standalone or against a pre-existing catalog.
GRANT USE CATALOG ON CATALOG ${CATALOG} TO `account users`;
GRANT USE SCHEMA ON SCHEMA ${CATALOG}.ai TO `account users`;

In [ ]:
%sql
-- Grant EXECUTE so the serving endpoint SP can call these tools at inference time.
GRANT EXECUTE ON FUNCTION ${CATALOG}.ai.get_order_details        TO `account users`;
GRANT EXECUTE ON FUNCTION ${CATALOG}.ai.get_order_delivery_time  TO `account users`;
GRANT EXECUTE ON FUNCTION ${CATALOG}.ai.get_location_timings     TO `account users`;

#### Model

In [0]:
%pip install -U -qqqq mlflow-skinny[databricks] "langgraph>=0.3.5,<0.4.0" databricks-langchain databricks-agents uv
dbutils.library.restartPython()

In [0]:
CATALOG = dbutils.widgets.get("CATALOG")
LLM_MODEL = dbutils.widgets.get("LLM_MODEL")

In [0]:
import re
from IPython.core.magic import register_cell_magic

@register_cell_magic
def writefilev(line, cell):
    """
    %%writefilev file.py
    Allows {{var}} substitutions while leaving normal {} intact.
    """
    filename = line.strip()

    def replacer(match):
        expr = match.group(1)
        return str(eval(expr, globals(), locals()))

    # Replace only double braces {{var}}
    content = re.sub(r"\{\{(.*?)\}\}", replacer, cell)

    with open(filename, "w") as f:
        f.write(content)
    print(f"Wrote file with substitutions: {filename}")

In [0]:
%%writefilev agent.py
from typing import Any, Generator, Literal, Optional, Sequence, Union

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    VectorSearchRetrieverTool,
    DatabricksFunctionClient,
    UCFunctionToolkit,
    set_uc_function_client,
)
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
try:
    from langgraph.graph.graph import CompiledGraph
    from langgraph.graph.state import CompiledStateGraph
except ImportError:  # langgraph >=0.4 restructured these internal modules
    from typing import Any
    CompiledGraph = Any
    CompiledStateGraph = Any
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentChunk,
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)

import json as _json
import uuid as _uuid
from pydantic import BaseModel, ValidationError

mlflow.langchain.autolog()

client = DatabricksFunctionClient()
set_uc_function_client(client)

class RefundDecision(BaseModel):
    refund_usd: float = 0.0
    refund_class: Literal["none", "partial", "full"] = "none"
    reason: str = ""


############################################
# Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = f"{{LLM_MODEL}}"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

system_prompt = """You are RefundGPT, a CX agent responsible for refund decisions on food delivery orders.

    You can call tools to gather the information you need. Start with an `order_id`.

    Instructions:
    1. Call `order_details(order_id)` first to get event history and confirm the id is valid and the order was delivered.
    2. Figure out the delivery duration by calling `get_order_delivery_time(order_id)`.
    3. Extract the location (either directly or from the first event's body).
    4. Call `get_location_timings(location)` to get the P50/P75/P99 values.
    5. Compare actual delivery time to those percentiles.

    Refund policy:

    A) SLA-based refund (primary path):
       - If the order arrived AFTER the P75 delivery time: recommend a `partial` or `full` refund based on how late.
       - If the order arrived BEFORE the P75: no SLA-based refund.

    B) Goodwill credit (only when complaint context is provided in the user message):
       The user may include lines such as:
           Customer complaint: "<text>"
           Complaint category: <category>
           Complaint agent suggested credit: $<amount>
       When all three are present AND the SLA path returns "none", you MAY ratify the
       complaint agent's goodwill credit:
       - Set `refund_class` = "partial"
       - Set `refund_usd` to the suggested credit amount (capped at $10)
       - In `reason`, note that the order was on time per SLA but a goodwill credit
         is being issued in response to the customer's complaint (cite the category).
       Only ratify when the suggested credit is plausible (>$0 and ≤$10) and the
       complaint category is non-empty. Otherwise return "none" with an SLA-based reason.

    When NO complaint context is provided, behave exactly as the SLA-based path (A) —
    do not invent goodwill credits.

    Output a single-line JSON with these fields:
    - `refund_usd` (float),
    - `refund_class` ("none" | "partial" | "full"),
    - `reason` (short human explanation. If goodwill, say so explicitly.)

    You must return only the JSON. No extra text or markdown."""

###############################################################################
## Define tools for your agent, enabling it to retrieve data or take actions
## beyond text generation
## To create and see usage examples of more tools, see
## https://docs.databricks.com/generative-ai/agent-framework/agent-tool.html
###############################################################################
tools = []

uc_tool_names = [f"{{CATALOG}}.ai.get_order_details", 
                 f"{{CATALOG}}.ai.get_location_timings",
                 f"{{CATALOG}}.ai.get_order_delivery_time"]
uc_toolkit = UCFunctionToolkit(function_names=uc_tool_names)
tools.extend(uc_toolkit.tools)

#####################
## Define agent logic
#####################


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[Sequence[BaseTool], ToolNode],
    system_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    # Define the function that determines which node to go to
    def should_continue(state: ChatAgentState):
        messages = state["messages"]
        last_message = messages[-1]
        # If there are function calls, continue. else, end
        if last_message.get("tool_calls"):
            return "continue"
        else:
            return "end"

    if system_prompt:
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}]
            + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: ChatAgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)

        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()


class LangGraphChatAgent(ChatAgent):
    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        request = {"messages": self._convert_messages_to_dict(messages)}

        result_messages = []
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                result_messages.extend(
                    ChatAgentMessage(**msg) for msg in node_data.get("messages", [])
                )
        for i in range(len(result_messages) - 1, -1, -1):
            msg = result_messages[i]
            role = msg.role if hasattr(msg, "role") else (msg.get("role") if isinstance(msg, dict) else None)
            content = msg.content if hasattr(msg, "content") else (msg.get("content", "") if isinstance(msg, dict) else "")
            if role == "assistant" and content:
                try:
                    parsed = RefundDecision.model_validate_json(content)
                    orig_id = getattr(msg, 'id', None) or str(_uuid.uuid4())
                    result_messages[i] = ChatAgentMessage(id=orig_id, role="assistant", content=parsed.model_dump_json())
                except (ValidationError, Exception):
                    pass
                break
        return ChatAgentResponse(messages=result_messages)

    def predict_stream(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> Generator[ChatAgentChunk, None, None]:
        request = {"messages": self._convert_messages_to_dict(messages)}
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                yield from (
                    ChatAgentChunk(**{"delta": msg}) for msg in node_data["messages"]
                )


# Create the agent object, and specify it as the agent object to use when
# loading the agent back for inference via mlflow.models.set_model()
agent = create_tool_calling_agent(llm, tools, system_prompt)
AGENT = LangGraphChatAgent(agent)
mlflow.models.set_model(AGENT)

In [0]:
import time

sample_order_id = None
for attempt in range(12):
    rows = spark.sql(f"""
        SELECT order_id 
        FROM {CATALOG}.lakeflow.all_events 
        WHERE event_type='delivered'
        LIMIT 1
    """).collect()
    if rows:
        sample_order_id = rows[0]['order_id']
        break
    print(f"No delivered events yet (attempt {attempt+1}/12). Waiting 30s for pipeline data...")
    time.sleep(30)

if not sample_order_id:
    raise RuntimeError(
        f"No delivered events found in {CATALOG}.lakeflow.all_events after 6 minutes. "
        "Ensure the Canonical_Data and Lakeflow pipeline stages completed and processed data."
    )

In [0]:
assert sample_order_id is not None
print(sample_order_id)

In [0]:
import mlflow
import sys
import os

sys.path.append(os.getcwd())

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_directory = os.path.dirname(notebook_path)
sys.path.append(project_directory)

from agent import LLM_ENDPOINT_NAME, tools
from databricks_langchain import VectorSearchRetrieverTool
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from pkg_resources import get_distribution
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)]
for tool in tools:
    resources.append(DatabricksFunction(function_name=tool.uc_function_name))

input_example = {
    "messages": [
        {
            "role": "user",
            "content": f"{sample_order_id}"
        }
    ]
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent_v2",
        python_model="agent.py",
        input_example=input_example,
        resources=resources,
        pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"mlflow=={get_distribution('mlflow').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            f"langgraph=={get_distribution('langgraph').version}",
        ],
    )

mlflow.set_active_model(model_id = logged_agent_info.model_id)

#### eval

In [0]:
# sample 10 order_ids
refund_queries = [
    row['order_id'] for row in spark.sql(f"""
        SELECT order_id 
        FROM {CATALOG}.lakeflow.all_events 
        WHERE event_type='delivered'
        LIMIT 10
    """).collect()
]

# wrap in correct input schema
data = []
for query in refund_queries:
    data.append(
        {
            "inputs": {
                "messages": [
                    {
                        "role": "user",
                        "content": query,
                    }
                ]
            },
        }
    )

print(data)

In [0]:
# create guideline, run evals

from mlflow.genai.scorers import Guidelines
import mlflow
import sys
import os
import time
import random

# Throttle MLflow eval concurrency to stay under the pay-per-token QPS limit
# on databricks-meta-llama-3-3-70b-instruct.  Defaults (10 data × 10 scorer)
# can easily exceed the per-workspace QPS for the shared endpoint.
#
# Pinned to 1 to match complaint_agent.  Even with the new depends_on edge
# serializing complaint after refunder, a single eval at WORKERS=2 still bursts
# above the workspace QPS ceiling on busy days (P1-9 / P1-9b).
os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"
os.environ["MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS"] = "1"

sys.path.append(os.getcwd())

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_directory = os.path.dirname(notebook_path)

# Add the project directory to the system path
sys.path.append(project_directory)

from agent import AGENT

refund_reason = Guidelines(
    name="refund_reason",
    guidelines=["If a refund is offered, its reason must relate to order timing, not to other issues such as missing components."]
)


def _is_rate_limit(exc: BaseException) -> bool:
    msg = str(exc)
    return (
        "REQUEST_LIMIT_EXCEEDED" in msg
        or "RateLimitError" in type(exc).__name__
        or "rate limit" in msg.lower()
    )


def predict_fn(messages):
    # Pay-per-token endpoints throttle aggressively.  Retry with jittered
    # exponential backoff so a single 429 doesn't fail the whole eval row.
    max_attempts = 6
    for attempt in range(max_attempts):
        try:
            return AGENT.predict({"messages": messages})
        except Exception as exc:
            if not _is_rate_limit(exc) or attempt == max_attempts - 1:
                raise
            backoff = min(60, 2 ** attempt) + random.uniform(0, 1)
            print(f"  ⚠️  rate-limited (attempt {attempt + 1}/{max_attempts}), sleeping {backoff:.1f}s")
            time.sleep(backoff)


# Eval idempotency: skip when the endpoint is already healthy and serving
# this exact UC model.  On a rate-limit-driven auto-retry, re-running the
# eval just re-burns workspace QPS budget against the same agent code (P1-9c).
# Helper is intentionally defined locally so this cell remains self-contained;
# P1-12 will unify with the identically-named helper used later for the
# register_model / agents.deploy guard.
from databricks.sdk import WorkspaceClient as _WorkspaceClient
from databricks.sdk.service.serving import EndpointStateReady as _EndpointStateReady


def _endpoint_already_serving(name: str, uc_model_name: str) -> bool:
    try:
        ep = _WorkspaceClient().serving_endpoints.get(name)
    except Exception:
        return False
    if not ep.state or ep.state.ready != _EndpointStateReady.READY:
        return False
    cfg = getattr(ep, "config", None) or getattr(ep, "pending_config", None)
    if not cfg:
        return False
    served = []
    for se in (getattr(cfg, "served_entities", None) or []):
        n = getattr(se, "entity_name", None)
        if n:
            served.append(n)
    for sm in (getattr(cfg, "served_models", None) or []):
        n = getattr(sm, "model_name", None)
        if n:
            served.append(n)
    return uc_model_name in served


_uc_model_for_eval = f"{CATALOG}.ai.refunder"
_endpoint_for_eval = dbutils.widgets.get("REFUND_AGENT_ENDPOINT_NAME")

# Gate 1 (hard kill): SKIP_EVAL=true short-circuits everything before we
# touch the LLM.  This is the lever that finally lets the bundle deploy run
# end-to-end without 429ing on the shared FMAPI endpoint — eval is the
# burst-iest LLM consumer in the bundle (~30-50 calls in tight succession)
# and is purely a quality-measurement step that the rest of the deploy
# does not depend on.  Defaults to "true" in databricks.yml; flip it to
# "false" when you specifically want a quality run.
try:
    _skip_eval = dbutils.widgets.get("SKIP_EVAL").strip().lower() == "true"
except Exception:
    _skip_eval = True  # default to skipping if the widget isn't plumbed yet

if _skip_eval:
    print(
        f"⏭  SKIP_EVAL=true — skipping mlflow.genai.evaluate to avoid the ~30-50 "
        f"LM-call eval burst.  Pass --params \"SKIP_EVAL=false\" to run eval."
    )
    results = None
elif _endpoint_already_serving(_endpoint_for_eval, _uc_model_for_eval):
    # Gate 2 (idempotency): even when eval is requested, skip when the
    # endpoint is already healthy and serving this exact UC model.  On a
    # rate-limit-driven auto-retry, re-running the eval just re-burns
    # workspace QPS budget against the same agent code.
    print(
        f"⏭  Endpoint {_endpoint_for_eval} already serving {_uc_model_for_eval}; "
        f"skipping eval to save ~3-5 min of LLM calls and avoid burning workspace "
        f"rate-limit budget on retry.  Delete the endpoint to force a fresh eval."
    )
    results = None
else:
    results = mlflow.genai.evaluate(
        data=data,
        scorers=[refund_reason],
        predict_fn=predict_fn,
    )

#### log refunder to `UC`

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateReady

mlflow.set_registry_uri("databricks-uc")

UC_MODEL_NAME = f"{CATALOG}.ai.refunder"
endpoint_name = dbutils.widgets.get("REFUND_AGENT_ENDPOINT_NAME")


def _endpoint_already_serving(name: str, uc_model_name: str) -> bool:
    """Return True iff a serving endpoint is READY and already serving uc_model_name.

    Used to short-circuit register_model + agents.deploy on re-runs of this
    stage when the endpoint from a previous run is still healthy — saves ~15
    minutes of cold container build + serving provisioning.  To force a fresh
    deploy after editing agent code, delete the endpoint and rerun the stage.
    """
    try:
        ep = WorkspaceClient().serving_endpoints.get(name)
    except Exception:
        return False
    if not ep.state or ep.state.ready != EndpointStateReady.READY:
        return False
    cfg = getattr(ep, "config", None) or getattr(ep, "pending_config", None)
    if not cfg:
        return False
    served = []
    for se in (getattr(cfg, "served_entities", None) or []):
        n = getattr(se, "entity_name", None)
        if n:
            served.append(n)
    for sm in (getattr(cfg, "served_models", None) or []):
        n = getattr(sm, "model_name", None)
        if n:
            served.append(n)
    return uc_model_name in served


_reuse_endpoint = _endpoint_already_serving(endpoint_name, UC_MODEL_NAME)

if _reuse_endpoint:
    print(
        f"\u267b\ufe0f Endpoint {endpoint_name} is already READY and serving {UC_MODEL_NAME}; "
        f"skipping register_model + agents.deploy (saves ~15 min). "
        f"Delete the endpoint to force a fresh deploy."
    )
    uc_registered_model_info = None
else:
    # register the model to UC
    uc_registered_model_info = mlflow.register_model(
        model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
    )

#### deploy the agent to model serving

In [0]:
from databricks import agents

if _reuse_endpoint:
    deployment_info = None
    print(f"\u2705 Endpoint {endpoint_name} is READY (reused from previous deploy)")
else:
    deployment_info = agents.deploy(
        model_name=UC_MODEL_NAME,
        model_version=uc_registered_model_info.version,
        scale_to_zero=False,
        endpoint_name=endpoint_name,
    )

In [0]:
print(deployment_info)

##### record model in state

In [ ]:
# Also add to UC-state — but only when we actually deployed a new endpoint.
# On the reuse path the endpoint was already registered by a previous run,
# so re-adding here would just create a duplicate uc_state row.
if deployment_info is not None:
    import sys
    sys.path.append('../utils')
    from uc_state import add

    add(dbutils.widgets.get("CATALOG"), "endpoints", deployment_info)
else:
    print("\u267b\ufe0f Endpoint already tracked in uc_state from a previous deploy; skipping add.")